In [30]:
import os
import json
import networkx as nx
from datetime import datetime, timezone
from dateutil.parser import parse
from collections import defaultdict

#Các hàm phân tích Thời gian (Time-Window)

In [31]:
def session_groups(alerts: list[dict], gap_sec: int = 120) -> list[list[dict]]:
    """Gom các alert xảy ra gần nhau về thời gian (cách nhau không quá gap_sec)."""
    if not alerts: 
        return []
        
    sorted_alerts = sorted(alerts, key=lambda a: a['ts'])
    groups = [[sorted_alerts[0]]]
    
    for alert in sorted_alerts[1:]:
        last_ts = parse(groups[-1][-1]['ts'])
        if (parse(alert['ts']) - last_ts).total_seconds() <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
            
    return groups

#Hàm phân tích Không gian (Topology) đã tối ưu

In [32]:
def topology_group(alerts, graph, max_hop=2):
    """Gom các alert dựa trên khoảng cách của chúng trên Service Graph."""
    if not alerts: 
        return []
        
    undirected = graph.to_undirected()
    by_service = defaultdict(list)
    for a in alerts: 
        by_service[a['service']].append(a)

    services = list(by_service.keys())
    service_set = set(services)
    parent = {s: s for s in services}

    # Disjoint Set Union (DSU)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry: 
            parent[rx] = ry

    # Tối ưu: Dùng BFS để tìm kiếm trong phạm vi max_hop
    for s in services:
        if s not in undirected: # Tránh lỗi nếu service không có trong graph
            continue
            
        reachable = nx.single_source_shortest_path_length(undirected, s, cutoff=max_hop)
        for node in reachable:
            if node in service_set and node != s:
                union(s, node)

    groups = defaultdict(list)
    for s in services:
        groups[find(s)].extend(by_service[s])
        
    return list(groups.values())

In [ ]:
def build_graph_from_json(filepath):
    """Xây dựng đồ thị có hướng từ file services.json."""
    with open(filepath, 'r') as f:
        data = json.load(f)
        
    G = nx.DiGraph()
    for edge in data.get('edges', []):
        G.add_edge(edge['from'], edge['to'])
    return G

current_dir = os.path.abspath('')
services_path = os.path.join(current_dir, 'dataset', 'services.json')
alerts_path = os.path.join(current_dir, 'dataset', 'alerts_sample.jsonl')

print("Đang đọc đồ thị topology...")
graph = build_graph_from_json(services_path)

print("Đang đọc danh sách cảnh báo...")
alerts = []
with open(alerts_path, 'r') as f:
    for line in f:
        alerts.append(json.loads(line.strip()))

print(f"Thành công! Đã nạp {len(alerts)} alerts và {graph.number_of_edges()} edges vào bộ nhớ.")

Đang đọc đồ thị topology...
Đang đọc danh sách cảnh báo...
Thành công! Đã nạp 20 alerts và 17 edges vào bộ nhớ.


In [ ]:
def correlate(alerts, graph, gap_sec=120, max_hop=2):
    """Hàm chạy luồng xử lý chính kết hợp Thời gian và Không gian."""
    sessions = session_groups(alerts, gap_sec=gap_sec)
    clusters = []
    
    for s_idx, session_alerts in enumerate(sessions):
        for g_idx, group in enumerate(topology_group(session_alerts, graph, max_hop)):
            if not group: 
                continue
            
            # Tạo danh sách vân tay (fingerprints) cho các alert
            fingerprints = {f"{a['service']}|{a['metric']}|{a['severity']}" for a in group}
            
            clusters.append({
                'cluster_id': f'c-{s_idx:03d}-{g_idx:03d}',
                'alert_count': len(group),
                'services': sorted({a['service'] for a in group}),
                'time_range': [min(a['ts'] for a in group), max(a['ts'] for a in group)],
                'max_severity': 'crit' if any(a['severity'] == 'crit' for a in group) else 'warn',
                'fingerprints': list(fingerprints)
            })
    return clusters

# 1. Thực thi thuật toán
final_clusters = correlate(alerts, graph, gap_sec=30, max_hop=2)

# 2. Tạo báo cáo chuẩn form yêu cầu
report = {
    "input_alerts": len(alerts),
    "output_clusters": len(final_clusters),
    "reduction_ratio": 1 - (len(final_clusters) / len(alerts) if alerts else 0),
    "clusters": final_clusters
}

# 3. Tạo thư mục results (nếu chưa có) và ghi file
os.makedirs(os.path.join(current_dir, 'results'), exist_ok=True)
results_path = os.path.join(current_dir, 'results', 'cluster_summary.json')

with open(results_path, 'w') as f:
    json.dump(report, f, indent=2)

# 4. In kết quả ra màn hình
print("==== KẾT QUẢ GOM CỤM (CORRELATION) ====")
print(f"- Alerts ban đầu: {report['input_alerts']}")
print(f"- Số cụm sinh ra: {report['output_clusters']}")
print(f"- Tỷ lệ giảm nhiễu: {report['reduction_ratio']:.2f} ({(report['reduction_ratio']*100):.1f}%)")
print(f"- Đã lưu kết quả tại: {results_path}")

==== KẾT QUẢ GOM CỤM (CORRELATION) ====
- Alerts ban đầu: 20
- Số cụm sinh ra: 5
- Tỷ lệ giảm nhiễu: 0.75 (75.0%)
- Đã lưu kết quả tại: d:\AWS\Xbrain\AIOps\w2\d1\results\cluster_summary.json
